### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="bank_marketing",
    dataset_year="2012",
    domain_str="finance",
    # Data Source<
    dataset_source="UCI",
    original_dataset_source_download_link="https://archive.ics.uci.edu/dataset/222/bank+marketing",
    download_description="""
We download the data from the UCI repository and uzip it to a predefined folder.

mkdir -p local-data-warehouse/bank_marketing/ && wget -P local-data-warehouse/bank_marketing/ https://archive.ics.uci.edu/static/public/222/bank+marketing.zip && unzip -o "local-data-warehouse/bank_marketing/bank+marketing.zip" -d local-data-warehouse/bank_marketing && unzip -o "local-data-warehouse/bank_marketing/bank.zip" -d local-data-warehouse/bank_marketing
""",
    # References
    academic_reference_bibtex="""@article{moro2014bank-marketing,
  title={A data-driven approach to predict the success of bank telemarketing},
  author={Moro, S{\'e}rgio and Cortez, Paulo and Rita, Paulo},
  journal={Decision Support Systems},
  volume={62},
  pages={22--31},
  year={2014},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="moro2014bank-marketing",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- We removed the "duration" feature following its original description to obtain a "realistic predictive model".
- We further remove the "month" and "day_of_week" features, as they also relate to the last contact -- which is not available in a real-world scenario.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="SubscribeTermDeposit",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="SubscribeTermDeposit",
)

## Preprocessing

In [2]:
import pandas as pd

data = pd.read_csv(f"{dataset_mold.path}/bank-full.csv", sep=";")

# Concatenate the two datasets
feature_names = [
    "age",
    "job",
    "marital",
    "education",
    "default",
    "balance",
    "housing",
    "loan",
    "contact",
    "day",
    "month",
    "duration",
    "campaign",
    "pdays",
    "previous",
    "poutcome",
    "SubscribeTermDeposit",
]

cat_features = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "campaign",
    "previous",
    "poutcome",
    "SubscribeTermDeposit",
]

data.columns = feature_names

df = data.sample(frac=1, random_state=42).reset_index(drop=True)

df[cat_features] = df[cat_features].astype("category") # NOTE: Unsure whether .astype("category") on the whole data is correct, since it defines cat codes using test data which is technically a leak

df = df.drop(columns=["day", "month", "duration"])

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 45,211
Columns: 14
Use sampling: False (sample size: 45,211)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['balance', 'pdays', 'age', 'campaign', 'previous', 'job', 'education', 'poutcome', 'marital', 'contact']
Rows remaining as candidates after top-10 filter: 1,018 (of 45,211)

#### Duplicate Report
Total duplicate rows: 274 (0.61% of dataset)
Duplicate rows ignoring target: 306 (0.68% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,age,job,marital,education,default,balance,housing,loan,contact,campaign,pdays,previous,poutcome,SubscribeTermDeposit
0,40,blue-collar,married,secondary,no,580,yes,no,unknown,1,-1,0,unknown,no
1,47,services,single,secondary,no,3644,no,no,unknown,2,-1,0,unknown,no
2,25,student,single,tertiary,no,538,yes,no,cellular,1,-1,0,unknown,no
3,42,management,married,tertiary,no,1773,no,no,cellular,1,336,1,failure,no
4,56,management,married,tertiary,no,217,no,yes,cellular,2,-1,0,unknown,no


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,job,category,0.0,0.0,12.0,"blue-collar, management, technician, admin., services, retired, self-employed, entrepreneur, unemployed, housemaid"
1,marital,category,0.0,0.0,3.0,"married, single, divorced"
2,education,category,0.0,0.0,4.0,"secondary, tertiary, primary, unknown"
3,default,category,0.0,0.0,2.0,"no, yes"
4,housing,category,0.0,0.0,2.0,"yes, no"
5,loan,category,0.0,0.0,2.0,"no, yes"
6,contact,category,0.0,0.0,3.0,"cellular, unknown, telephone"
7,campaign,category,0.0,0.0,48.0,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10"
8,previous,category,0.0,0.0,41.0,"0, 1, 2, 3, 4, 5, 6, 7, 8, 9"
9,poutcome,category,0.0,0.0,4.0,"unknown, failure, other, success"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
age,45211.0,40.936210,10.618762,18.0,95.0
balance,45211.0,1362.272058,3044.765829,-8019.0,102127.0
pdays,45211.0,40.197828,100.128746,-1.0,871.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column               rank                           
SubscribeTermDeposit 1              no  39922  88.30
                     2             yes   5289  11.70
campaign             1               1  17544  38.80
                     2               2  12505  27.66
                     3               3   5521  12.21
                     4               4   3522   7.79
                     5               5   1764   3.90
contact              1        cellular  29285  64.77
                     2         unknown  13020  28.80
                     3       telephone   2906   6.43
default              1              no  44396  98.20
                     2             yes    815   1.80
education            1       secondary  23202  51.32
                     2        tertiary  13301  29.42
                     3         primary   6851  15.15
                     4         unknown   1857   4.11
housing              1             yes  25130  55.58
                     2              no  20081  44.42
job                  1     blue-collar   9732  21.53
                     2      management   9458  20.92
                     3      technician   7597  16.80
                     4          admin.   5171  11.44
                     5        services   4154   9.19
loan                 1              no  37967  83.98
                     2             yes   7244  16.02
marital              1         married  27214  60.19
                     2          single  12790  28.29
                     3        divorced   5207  11.52
poutcome             1         unknown  36959  81.75
                     2         failure   4901  10.84
                     3           other   1840   4.07
                     4         success   1511   3.34
previous             1               0  36954  81.74
                     2               1   2772   6.13
                     3               2   2106   4.66
                     4               3   1142   2.53
                     5               4    714   1.58

In [8]:
# Target Distribution
target_df

,count,pct
SubscribeTermDeposit,,
no,39922,88.3
yes,5289,11.7


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to bank_marketing/019d9cc3-2925-71e4-b7fd-976c8c779195
019d9cc3-2925-71e4-b7fd-976c8c779195
2191763cc62ba975bf860bbd7d3e8d0cae00a614edfdf00ebfaf6290381f6db9
